In [ ]:
import pandas
import matplotlib.pyplot as pyplot
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.naive_bayes import BernoulliNB
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
clean_dataset = pandas.read_csv('df_banco_final.csv', sep=';')

In [ ]:
y = clean_dataset['default']
x = clean_dataset.drop('default', axis=1)

In [ ]:
x.head()

In [ ]:

normalizer = StandardScaler()
x_normalized = normalizer.fit_transform(x)

In [ ]:
x_normalized

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_normalized, y, test_size=0.3, random_state=123)

In [ ]:
mediana = np.median(x_train)
mediana

In [ ]:
x_train_binarizado = np.where(x_train > mediana, 1, 0)
x_train_binarizado

In [ ]:
#entrenando el modelo
bnb = BernoulliNB()
bnb.fit(x_train_binarizado, y_train)

In [ ]:
#Binarizando la base de prueba
X_test_binarizado = np.where(x_test > np.median(x_test), 1, 0)
X_test_binarizado

In [ ]:
predictions = bnb.predict(X_test_binarizado)
predictions

## Evaluamos el modelo

### Matriz de confusi?n

In [ ]:
confmatrix = confusion_matrix(y_test, predictions)
confmatrix

visual_confmatrix = ConfusionMatrixDisplay(confmatrix)
visual_confmatrix.plot(cmap='PuBu')

In [ ]:
confmatrix

In [ ]:
tn = confmatrix[0][0]
fp = confmatrix[0][1]
specificity = tn/(tn + fp)
specificity

In [ ]:
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)
rocauc = roc_auc_score(y_test, predictions)


In [ ]:
model_scores = {
  'model': 'Naive_Bayes(Bernulli_NB)',
  'accuracy': accuracy,
  'precision': precision,
  'recall': recall,
  'f1_score': f1,
  'roc_auc_score': rocauc,
  'specificity' : specificity
}
model_scores

In [ ]:
def save_scores(model_scores: dict):
  status = []
  #Utilizamos un try-except en caso de que el archivo no exista que cree uno
  try:
    #Si el archivo existe contin?a por ac?
    scores = pandas.read_csv('scores.csv', sep=';')
    status.append('El archivo existe')
    exist = model_scores['model'] in scores['model'].values
    #Validamos si el modelo ya existe en el archivo de scores
    if (exist):
      #Si el modelo ya existe reemplazamos sus valores
      status.append('El modelo existe')
      criteria = scores['model'] == model_scores['model']
      index = scores[criteria].index[0]
      scores.iloc[index] = model_scores
      status.append('Se reemplazaron los valores del modelo')
    else:
      #Si el modelo no existe lo agregamos
      status.append('El modelo no existe')
      df_model_scores = pandas.DataFrame(model_scores, index=[0])
      scores = pandas.concat([scores, df_model_scores], ignore_index=True)
      status.append('Se a?adi? modelo nuevo y sus valores')
  except:
    #Si no existe el archivo lo creamos y cargamos los scores del modelo
    status.append('El archivo no existe')
    scores = pandas.DataFrame(model_scores, index=[0])

  status.append('Se sobrescrive el archivo scores.csv con valores nuevos')
  scores.to_csv('scores.csv', sep=';', index=False)
  return status

In [ ]:
save_scores(model_scores)

In [ ]:
#instancia de kfolds
kfolds = KFold(n_splits=5, shuffle=True)

In [ ]:
#aplicaci?n de Cross validation
accuracy_crossval = cross_val_score(
  estimator=bnb,
  X=x_train_binarizado,
  y=y_train,
  scoring='accuracy',
  cv=kfolds
)
accuracy_crossval.mean()

In [ ]:
bnb_hiperparams = {
  'alpha': [1e-10, 0.1, 0.5, 1.0, 1.5, 2.0],  
  'force_alpha': [False,True],
  'binarize': [None, 0.0, 0.5], 
  'fit_prior': [True, False]
}
# 'class_prior': [None, [0.3, 0.7], [0.2, 0.8]],

In [ ]:
randomsearch_bnb_model = RandomizedSearchCV(
  bnb,
  bnb_hiperparams,
  n_iter=72,
  cv=kfolds,
  scoring='accuracy',
  n_jobs=-1
)

In [ ]:
randomsearch_fit = randomsearch_bnb_model.fit(x_train_binarizado, y_train)

In [ ]:
randomsearch_fit.best_score_

In [ ]:
randomsearch_fit.best_params_

In [ ]:
bnb_optimized = BernoulliNB(force_alpha = False,
 fit_prior =False,
 binarize = None,
 alpha = 1e-10)
bnb_optimized.fit(x_train_binarizado, y_train)
bnb_optimized_predictions = bnb_optimized.predict(X_test_binarizado)

In [ ]:
confmatrix = confusion_matrix(y_test, bnb_optimized_predictions)
confmatrix

visual_confmatrix = ConfusionMatrixDisplay(confmatrix)
visual_confmatrix.plot(cmap='PuBu')

In [ ]:
accuracy = accuracy_score(y_test, bnb_optimized_predictions)
precision = precision_score(y_test, bnb_optimized_predictions)
recall = recall_score(y_test, bnb_optimized_predictions)
f1 = f1_score(y_test, bnb_optimized_predictions)
rocauc = roc_auc_score(y_test, bnb_optimized_predictions)
tn = confmatrix[0][0]
fp = confmatrix[0][1]
specificity = tn/(tn + fp)

In [ ]:
accuracy

In [ ]:
model_scores = {
  'model': 'Naive_Bayes(Bernulli_NB)_optimized',
  'accuracy': accuracy,
  'precision': precision,
  'recall': recall,
  'f1_score': f1,
  'roc_auc_score': rocauc,
  'specificity' : specificity
}
model_scores

In [ ]:
save_scores(model_scores)